# A certified upper bound for the growth constant

Companion to Section IV-B of *Asymptotic Growth of the Number of Rubik's Snake Shapes*. This notebook is standalone; it uses no unpublished files. Dependencies: `../rubiks_snake.py`, NumPy, Numba, SciPy and the standard library, plus a Jupyter runtime.

Vertices are all valid rotation words of length $m$. Every valid word of length $m+1$ gives an edge from its prefix to its suffix. With **column convention** $A_{v,u}=\#(u\to v)$, every globally valid word gives a graph path, so $\mu\leq\rho(A_m)$.

Strongly connected components (SCCs) give a block-triangular matrix. Its spectral radius is the maximum over diagonal blocks. We retain internal edges of every cyclic component, including singleton self-loops, and discard only zero diagonal blocks. Iteration with $A+I$, normalized independently in each SCC, proposes a vector. After rounding to a positive integer vector, an **exact overflow-free check** of $D A v\leq p v$ certifies $\rho(A)\leq p/D$. Floating-point iteration alone is not a proof.

Outputs through $m=11$ were reproduced from these cell sources on 2026-09-19 (Python 3.14.6, NumPy 2.5.3, Numba 0.67.0, SciPy 1.18.1). The final $m=13$ output is an **archived research result from 2026-09-11**, not a fresh execution. Execution counts remain null because validation used a script rather than an editor kernel. Rerun to regenerate all windows and verify certificates with the current implementation. The old implementation's direct fixed-width products have been replaced by overflow-free comparisons.

**Resource warning:** $m=13$ previously needed about 195 seconds and 6.5 GiB; current peak memory and timing may differ. Do not execute the final large cell on a memory-constrained machine. Larger $m$ grows exponentially. The enumerator's formal limit is $m\leq18$, not a promise that such runs fit in memory.

In [ ]:
from pathlib import Path
from time import perf_counter
import sys
started = perf_counter()
cwd = Path.cwd()
candidates = (cwd, cwd / 'rubiks-snake', cwd.parent, cwd.parent / 'rubiks-snake')
snake_dir = next((p for p in candidates if (p / 'rubiks_snake.py').is_file()), None)
if snake_dir is None:
    raise FileNotFoundError('Run from asymptotic-analysis, rubiks-snake, or the repository root')
sys.path.insert(0, str(snake_dir.resolve()))
import numpy as np
import numba
import scipy
from rubiks_snake import window_upper_bound
print(f'Python {sys.version.split()[0]}; NumPy {np.__version__}; Numba {numba.__version__}; SciPy {scipy.__version__}')
print(f'Setup: {perf_counter() - started:.3f} s')

In [ ]:
def get_bound(m, iterations=500, scale=10**12, denominator=10**9):
    """Construct the window graph and return an exactly verified spectral bound."""
    started = perf_counter()
    result = window_upper_bound(m, iterations, scale, denominator)
    result['seconds'] = perf_counter() - started
    q = result['bound']
    print(f"m={m}: {result['states']} cyclic states, {result['edges']} internal edges")
    print(f"mu <= {q} = {float(q):.9f}; exact certificate passed; {result['seconds']:.3f} s")
    return result

## Small examples

Change `m` to retain a longer collision window. More iterations or a larger `scale` can improve the certificate vector; `denominator` controls rational resolution. These numerical knobs do not affect validity: every returned bound passes the exact check.

In [ ]:
SMALL_WINDOWS = (3, 5, 7)
small_results = [get_bound(m) for m in SMALL_WINDOWS]

m=3: 59 cyclic states, 225 internal edges
mu <= 3810528899/1000000000 = 3.810528899; exact certificate passed; 0.024 s
m=5: 822 cyclic states, 3062 internal edges
mu <= 1860055621/500000000 = 3.720111242; exact certificate passed; 0.007 s
m=7: 11293 cyclic states, 41819 internal edges
mu <= 370307551/100000000 = 3.703075510; exact certificate passed; 0.074 s


## Published certificates

The first two runs are much smaller than the final, separately selectable cell. Edit `PUBLISHED_RUNS` or `LARGE_WINDOW` to change the computation. The tuples specify `(m, iterations)`. Only the last computation requires the historical 6.5 GiB scale of memory. The output counts refer to the cyclic diagonal blocks, not the full graph; these certificates alone do **not** provide a pointwise exponential prefactor.

In [ ]:
PUBLISHED_RUNS = [(9, 500), (11, 500)]
published_results = [get_bound(m, iterations) for m, iterations in PUBLISHED_RUNS]

m=9: 153306 cyclic states, 565050 internal edges
mu <= 1842734183/500000000 = 3.685468366; exact certificate passed; 1.381 s
m=11: 2070446 cyclic states, 7608583 internal edges
mu <= 734968961/200000000 = 3.674844805; exact certificate passed; 26.251 s


### Largest published run: several GiB of memory

This is an actual runnable computation, not a lookup of the stored result. Skip this cell if insufficient memory is available. Its retained output is archival, pending a fresh run of the overflow-safe certificate check.

In [ ]:
LARGE_WINDOW, LARGE_ITERATIONS = 13, 180
largest = get_bound(LARGE_WINDOW, LARGE_ITERATIONS)

Archived reference result (2026-09-11; original implementation):
m=13: 27847372 cyclic states, 102127233 internal edges; mu <= 3.667542939
Historical elapsed time: 194.79 s; peak memory: 6828692 kB
